# Día 9 · Unidad 9 — Fuentes de datos (III): APIs REST con `requests`

**Objetivos de aprendizaje**
- Entender qué es una API REST y la estructura de una llamada HTTP (verbo, URL, headers, params, body).
- Usar `requests` para hacer peticiones GET reales y trabajar con la respuesta JSON.
- Gestionar errores HTTP (`401 Unauthorized`, `404 Not Found`) de forma explícita.
- Usar credenciales/tokens sin escribirlos nunca directamente en el código, mediante variables de entorno y `python-dotenv`.
- Entender por qué un archivo `.env` con secretos reales nunca se commitea a un repositorio.

**Duración estimada:** 75-90 minutos.

**Requisitos previos:** `02_teoria_bbdd_crud.ipynb` de este mismo día.

**Nota técnica:** para que este notebook sea reproducible sin depender de internet (ni de que un servicio externo siga activo durante el curso), en vez de consultar una API real montamos un **servidor HTTP mínimo, local**, dentro del propio notebook. Las llamadas con `requests` son reales — van por el protocolo HTTP de verdad — simplemente el servidor al otro lado también es nuestro.


## 1. Qué es una API REST, en cuatro ideas

Una **API** (*Application Programming Interface*) es una forma de que dos programas se comuniquen entre sí. Una **API REST** es un estilo de API muy extendido que usa el protocolo HTTP — el mismo que usa vuestro navegador para cargar páginas web — como transporte.

Cada llamada a una API REST se compone de:

- **Verbo HTTP**: qué acción se pide. `GET` (leer), `POST` (crear), `PUT`/`PATCH` (actualizar), `DELETE` (eliminar) — el mismo vocabulario del CRUD que acabáis de ver con SQLModel, pero a través de la red.
- **URL**: qué recurso se pide, p. ej. `http://api.ejemplo.com/polizas/POL100000`.
- **Headers**: metadatos de la petición — autenticación, formato esperado de la respuesta, etc.
- **Params** (opcional): filtros o parámetros añadidos a la URL, p. ej. `?producto=AUTO`.
- **Body** (opcional, normalmente en `POST`/`PUT`): los datos que se envían, casi siempre en formato **JSON**.

La respuesta trae un **código de estado HTTP** (`200` = todo bien, `401` = no autenticado, `404` = no existe, `500` = error del servidor...) y normalmente un cuerpo, también en JSON.

> 📊 En Excel esto sería como usar `SERVICIOWEB()` o Power Query "Desde web": le dais una URL (y, si hace falta, unas credenciales) y Excel os trae los datos de vuelta. `requests` es la forma programática de hacer exactamente eso, con control total sobre la petición.


## 2. Un servidor HTTP local para practicar sin depender de internet

Vamos a levantar, dentro de este mismo notebook, un servidor HTTP muy simple que expone un resumen de `data/raw/cartera_polizas.csv` como JSON — imaginad que es "la API de pólizas" de una aseguradora ficticia. Usamos únicamente el módulo estándar `http.server` (no hace falta instalar Flask ni FastAPI para este ejemplo mínimo).

El servidor:
- Sirve `GET /polizas` (lista resumida) y `GET /polizas/<poliza_id>` (una póliza concreta).
- Exige una cabecera `Authorization: Bearer <token>` — si falta o el token no coincide, responde `401 Unauthorized`.
- Si se pide un `poliza_id` que no existe, responde `404 Not Found`.
- Arranca en un **hilo de fondo** (`threading.Thread(daemon=True)`), para no bloquear el notebook, y en un **puerto elegido por el sistema operativo** (`port=0`) para no chocar nunca con "puerto ya en uso" si el notebook se ejecuta varias veces seguidas.


In [1]:
import json
import threading
from http.server import BaseHTTPRequestHandler, HTTPServer
from urllib.parse import urlparse

import pandas as pd

# Preparamos el "dataset" que servirá la API: un resumen de la cartera de pólizas
cartera = pd.read_csv("../data/raw/cartera_polizas.csv")
polizas_api = (
    cartera[["poliza_id", "producto", "provincia", "prima_anual", "anulada"]]
    .head(50)
    .to_dict(orient="records")
)
polizas_por_id = {p["poliza_id"]: p for p in polizas_api}

TOKEN_VALIDO = "demo-token-CAMBIAME"  # token ficticio -- en el apartado 4 veremos cómo NO escribirlo así


def crear_handler(token_esperado: str):
    # Fábrica de handler: así el token se puede inyectar sin usar una variable global mutable.

    class PolizasAPIHandler(BaseHTTPRequestHandler):
        def _responder_json(self, status: int, payload: dict) -> None:
            cuerpo = json.dumps(payload, ensure_ascii=False).encode("utf-8")
            self.send_response(status)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(cuerpo)))
            self.end_headers()
            self.wfile.write(cuerpo)

        def _autenticado(self) -> bool:
            auth = self.headers.get("Authorization", "")
            return auth == f"Bearer {token_esperado}"

        def do_GET(self):  # noqa: N802 (nombre exigido por BaseHTTPRequestHandler)
            if not self._autenticado():
                self._responder_json(401, {"error": "Falta token válido en la cabecera Authorization"})
                return

            ruta = urlparse(self.path).path

            if ruta == "/polizas":
                self._responder_json(200, {"polizas": polizas_api, "total": len(polizas_api)})
            elif ruta.startswith("/polizas/"):
                poliza_id = ruta.removeprefix("/polizas/")
                poliza = polizas_por_id.get(poliza_id)
                if poliza is None:
                    self._responder_json(404, {"error": f"No existe la póliza {poliza_id!r}"})
                else:
                    self._responder_json(200, poliza)
            else:
                self._responder_json(404, {"error": f"Ruta no reconocida: {ruta}"})

        def log_message(self, format, *args):  # noqa: A002 (firma exigida por la clase base)
            pass  # silencia el log por defecto de BaseHTTPRequestHandler, muy verboso en un notebook

    return PolizasAPIHandler


servidor = HTTPServer(("localhost", 0), crear_handler(TOKEN_VALIDO))  # port=0 -> el SO asigna un puerto libre
puerto = servidor.server_address[1]  # puerto real asignado, nunca lo fijamos a mano

hilo_servidor = threading.Thread(target=servidor.serve_forever, daemon=True)
hilo_servidor.start()

print(f"Servidor de prueba escuchando en http://localhost:{puerto}")


Servidor de prueba escuchando en http://localhost:56970


> ⚠️ **Error típico**: fijar el puerto a mano (p. ej. `HTTPServer(("localhost", 8765), ...)`). Si el notebook se ejecuta dos veces seguidas sin reiniciar el kernel, o si hay otro proceso usando ese puerto, la segunda ejecución falla con `OSError: [Errno 98] Address already in use`. Usando `port=0` el sistema operativo asigna siempre un puerto libre, y `server.server_address[1]` os dice cuál fue — el notebook se vuelve idempotente sin ningún truco adicional.


## 3. La primera llamada real con `requests`

Con el servidor ya escuchando, hacemos una petición `GET` de verdad — pasa por el stack de red del sistema operativo, exactamente igual que si `localhost` fuera un servidor remoto.


In [2]:
import requests

base_url = f"http://localhost:{puerto}"

respuesta = requests.get(
    f"{base_url}/polizas",
    headers={"Authorization": f"Bearer {TOKEN_VALIDO}"},
)

print("Código de estado:", respuesta.status_code)
print("Content-Type:", respuesta.headers["Content-Type"])

datos = respuesta.json()  # requests parsea el JSON de la respuesta a dict/list de Python
print("Total de pólizas devueltas:", datos["total"])
pd.DataFrame(datos["polizas"]).head()


Código de estado: 200
Content-Type: application/json; charset=utf-8
Total de pólizas devueltas: 50


,poliza_id,producto,provincia,prima_anual,anulada
0,POL100000,AUTO,Alicante,455.69,True
1,POL100001,HOGAR,Madrid,274.72,True
2,POL100002,HOGAR,Valencia,207.32,False
3,POL100003,AUTO,Vizcaya,540.71,False
4,POL100004,AUTO,Vizcaya,589.46,False


`respuesta.json()` es el equivalente de `json.loads(respuesta.text)`, pero más cómodo — convierte directamente el cuerpo de la respuesta (una cadena de texto en formato JSON) a estructuras nativas de Python (`dict`, `list`, `str`, `float`, `bool`, `None`).

### Params: filtrar con la URL

Para pedir un recurso concreto, se construye la URL con el identificador — es el patrón habitual en APIs REST (`/polizas/<id>`):


In [3]:
poliza_id_ejemplo = polizas_api[0]["poliza_id"]

respuesta = requests.get(
    f"{base_url}/polizas/{poliza_id_ejemplo}",
    headers={"Authorization": f"Bearer {TOKEN_VALIDO}"},
)

print(respuesta.status_code)
respuesta.json()


200


{'poliza_id': 'POL100000',
 'producto': 'AUTO',
 'provincia': 'Alicante',
 'prima_anual': 455.69,
 'anulada': True}

## 4. Manejo de errores HTTP: 401 y 404

Una llamada a una API real falla constantemente por motivos legítimos: token caducado, recurso que no existe, límite de peticiones superado... `requests` **no lanza una excepción automáticamente** cuando el servidor responde con un código de error (`4xx`/`5xx`) — hay que comprobar `status_code` explícitamente, o usar `raise_for_status()`.


In [4]:
# 401: sin cabecera Authorization
respuesta_sin_token = requests.get(f"{base_url}/polizas")
print("Sin token ->", respuesta_sin_token.status_code, respuesta_sin_token.json())

# 401: con token incorrecto
respuesta_token_malo = requests.get(
    f"{base_url}/polizas", headers={"Authorization": "Bearer token-equivocado"}
)
print("Token incorrecto ->", respuesta_token_malo.status_code, respuesta_token_malo.json())

# 404: póliza que no existe
respuesta_404 = requests.get(
    f"{base_url}/polizas/POL_NO_EXISTE", headers={"Authorization": f"Bearer {TOKEN_VALIDO}"}
)
print("Póliza inexistente ->", respuesta_404.status_code, respuesta_404.json())


Sin token -> 401 {'error': 'Falta token válido en la cabecera Authorization'}


Token incorrecto -> 401 {'error': 'Falta token válido en la cabecera Authorization'}


Póliza inexistente -> 404 {'error': "No existe la póliza 'POL_NO_EXISTE'"}


In [5]:
def obtener_poliza(poliza_id: str) -> dict | None:
    '''Consulta una póliza en la API, gestionando los códigos de error explícitamente.'''
    respuesta = requests.get(
        f"{base_url}/polizas/{poliza_id}",
        headers={"Authorization": f"Bearer {TOKEN_VALIDO}"},
    )

    if respuesta.status_code == 200:
        return respuesta.json()
    if respuesta.status_code == 401:
        raise PermissionError("Token inválido o ausente: revisa tus credenciales.")
    if respuesta.status_code == 404:
        print(f"Aviso: la póliza {poliza_id!r} no existe en la API.")
        return None

    # cualquier otro código: lo tratamos como error inesperado
    respuesta.raise_for_status()


print(obtener_poliza(poliza_id_ejemplo))
print(obtener_poliza("POL_NO_EXISTE"))


{'poliza_id': 'POL100000', 'producto': 'AUTO', 'provincia': 'Alicante', 'prima_anual': 455.69, 'anulada': True}


Aviso: la póliza 'POL_NO_EXISTE' no existe en la API.
None


> ⚠️ **Error típico**: asumir que si `requests.get(...)` no lanzó una excepción, la petición "salió bien". Una respuesta `404` o `401` es una respuesta HTTP perfectamente válida — `requests` la devuelve con normalidad, no la convierte en excepción. Hay que mirar `status_code` (o llamar a `.raise_for_status()`, que sí lanza `requests.exceptions.HTTPError` para códigos `4xx`/`5xx`) para saber si la petición tuvo éxito de verdad.

**Para ti:** modifica `obtener_poliza` (o escribe una función nueva) para que, en vez de un único `poliza_id`, acepte una lista y devuelva un DataFrame con las pólizas encontradas, imprimiendo un aviso por cada `poliza_id` que dé 404 — sin que el proceso se detenga en el primer error.


In [6]:
# Tu código aquí


## 5. Credenciales y variables de entorno: nunca un token en el código

En todo lo anterior, `TOKEN_VALIDO = "demo-token-CAMBIAME"` está escrito **directamente en el notebook** — aceptable para un ejemplo didáctico con un token inventado, pero es exactamente lo que **no** se debe hacer con un token real: si el notebook se sube a un repositorio (incluso privado), el token queda expuesto en el historial de git para siempre, aunque luego se borre de la última versión.

La práctica estándar es usar **variables de entorno**: el token vive fuera del código, en el entorno del sistema operativo (o en un archivo local que nunca se commitea), y el código solo lo *lee* en tiempo de ejecución.

`python-dotenv` es una librería pequeña que carga variables de entorno desde un archivo `.env` a `os.environ`. El patrón habitual en cualquier proyecto:

- `.env.example` (**sí** se commitea): documenta qué variables hacen falta, con valores de ejemplo evidentemente ficticios.
- `.env` (**nunca** se commitea — debe estar en `.gitignore`): la copia real, con los valores reales, que cada persona/entorno rellena por su cuenta.

Ya hemos creado `.env.example` en la carpeta de este día (mirad el archivo junto a este notebook). Su contenido es:

```
# Copia este archivo a .env y rellena con tus valores reales.
# .env NUNCA se commitea (está en .gitignore) -- .env.example sí, como documentación.
API_TOKEN=demo-token-CAMBIAME
```


In [7]:
from pathlib import Path

env_example = Path("../dia_09_datos_y_estadistica/.env.example")
print(env_example.read_text(encoding="utf-8"))


# Copia este archivo a .env y rellena con tus valores reales.
# .env NUNCA se commitea (está en .gitignore) -- .env.example sí, como documentación
# de qué variables necesita el proyecto.
API_TOKEN=demo-token-CAMBIAME



Para que este notebook sea ejecutable de principio a fin sin que cada persona tenga que crear su propio `.env` a mano antes de empezar, simulamos el paso "cada desarrollador copia `.env.example` a `.env` y lo rellena" escribiendo un `.env` **temporal** (fuera del repositorio, en una carpeta de temporales del sistema) con el mismo token ficticio. Es exactamente el mismo mecanismo (`load_dotenv` + `os.getenv`) que usaríais con un `.env` real y secretos reales — el token en sí sigue siendo de mentira.


In [8]:
import os
import tempfile
from dotenv import load_dotenv

# Simulación de "tu .env local, nunca commiteado" -- vive en una carpeta temporal del sistema,
# fuera del repositorio, exactamente igual que debería vivir un .env real en tu máquina.
directorio_temporal = tempfile.mkdtemp(prefix="curso_cac_env_")
ruta_env_temporal = Path(directorio_temporal) / ".env"
ruta_env_temporal.write_text("API_TOKEN=demo-token-CAMBIAME\n", encoding="utf-8")

load_dotenv(dotenv_path=ruta_env_temporal)  # carga las variables del .env a os.environ
token_desde_entorno = os.getenv("API_TOKEN")

print(f"Token leído desde la variable de entorno: {token_desde_entorno!r}")
print(f"¿Coincide con el token que usa el servidor de prueba?: {token_desde_entorno == TOKEN_VALIDO}")


Token leído desde la variable de entorno: 'demo-token-CAMBIAME'
¿Coincide con el token que usa el servidor de prueba?: True


In [9]:
# Usamos el token leído del entorno, NO una constante escrita a mano, para llamar a la API
respuesta = requests.get(
    f"{base_url}/polizas",
    headers={"Authorization": f"Bearer {token_desde_entorno}"},
)
print(respuesta.status_code, "- total pólizas:", respuesta.json()["total"])


200 - total pólizas: 50


> ⚠️ **Error típico**: commitear un `.env` real "solo esta vez, para probar en clase" o "porque total el repo es privado". No hay excepción segura a esta regla — el historial de git conserva todo lo commiteado, aunque se borre después en un commit posterior; y un repo "privado hoy" puede no serlo mañana. La forma correcta de compartir credenciales del equipo, si hace falta, es un gestor de secretos (Vault, AWS Secrets Manager, variables de entorno del propio servidor de despliegue...), nunca un archivo en el repositorio.

Recordad también `docs/uso_ia_en_el_curso.md`: nunca peguéis un token o una contraseña real dentro de un prompt a un asistente de IA — tratad cualquier IA en la nube como un tercero externo, igual que tratad un `.env`: fuera del control de versiones, fuera de cualquier sitio que no controléis vosotros.


## 6. Cerrar el servidor: idempotencia

Por buena práctica (y porque el enunciado de este notebook lo exige explícitamente), cerramos el servidor de prueba al terminar. Así, si alguien ejecuta el notebook completo varias veces seguidas en el mismo kernel, cada ejecución arranca un servidor nuevo en un puerto nuevo, sin interferir con instancias previas.


In [10]:
servidor.shutdown()  # detiene el bucle serve_forever()
servidor.server_close()  # libera el socket del puerto
print(f"Servidor en el puerto {puerto} cerrado correctamente.")


Servidor en el puerto 56970 cerrado correctamente.


## Resumen

- Una API REST se llama con un **verbo HTTP** (`GET`/`POST`/`PUT`/`DELETE`) contra una **URL**, con **headers** y, si aplica, **params**/**body** — la respuesta trae un **código de estado** y normalmente un cuerpo en **JSON**.
- Para practicar sin depender de internet ni de un servicio externo, se puede montar un servidor HTTP mínimo local con `http.server.HTTPServer`, en un hilo de fondo (`daemon=True`) y en un puerto elegido por el sistema (`port=0` + `server.server_address[1]`) para que el notebook sea idempotente.
- `requests.get(url, headers=..., params=...)` hace la petición; `.status_code` da el código HTTP; `.json()` parsea el cuerpo de la respuesta a estructuras de Python.
- `requests` **no lanza excepción** por un `4xx`/`5xx` automáticamente — hay que comprobar `status_code` explícitamente o llamar a `.raise_for_status()`.
- Las credenciales (tokens, contraseñas, claves de conexión) nunca se escriben directamente en el código: se leen de **variables de entorno**, típicamente cargadas desde un archivo `.env` con `python-dotenv`.
- `.env.example` (con valores ficticios) sí se commitea, como documentación; `.env` (con valores reales) nunca se commitea — debe estar en `.gitignore`.
- Cerrar explícitamente un servidor de pruebas (`server.shutdown()` + `server.server_close()`) al final de un notebook lo hace seguro de re-ejecutar tantas veces como haga falta.

**Siguiente:** `04_teoria_estadistica_descriptiva.ipynb` — arranca la Unidad 10, la parte de estadística aplicada del día.
